In [ ]:
import kagglehub


# Download latest version
path = kagglehub.dataset_download("donkeys/retinopathy-train-2015")

print("Path to dataset files:", path)

In [1]:
### **Preprocessing, GPU-Accelerated Augmentation & tf.data Setup**

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess_input


# 0) Paths / Load CSV
BASE_DIR = "/home/jovyan/.cache/kagglehub/datasets/donkeys/retinopathy-train-2015/versions/1"
IMG_DIR = os.path.join(BASE_DIR, "rescaled_train_896")
CSV_PATH = os.path.join(BASE_DIR, "trainLabels.csv")

df = pd.read_csv(CSV_PATH)

# Setting three possible names for each label  
lower_cols = {c.lower(): c for c in df.columns}
img_col = lower_cols.get("image") or lower_cols.get("id_code") or df.columns[0]
label_col = lower_cols.get("level") or lower_cols.get("diagnosis") or df.columns[1]

# Craeting full paths
df["path"] = df[img_col].astype(str).apply(lambda x: os.path.join(IMG_DIR, f"{x}.png"))
df["label"] = df[label_col].astype(int)

# In case there are some existing files we keep them
exists_mask = df["path"].apply(os.path.exists)
df = df.loc[exists_mask].reset_index(drop=True)

X = np.array(df["path"].values, dtype=object)
y = np.array(df["label"].values, dtype=int)


# 1) Split 70/15/15 (stratified)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp
)

print("Split sizes:", len(X_train), len(X_val), len(X_test))


# 2) Preprocessing building blocks (GPU-friendly)
def decode_png(path: tf.Tensor) -> tf.Tensor:
    b = tf.io.read_file(path)
    img = tf.image.decode_png(b, channels=3)  # uint8 (H,W,3)
    return img

def central_square_crop(img: tf.Tensor) -> tf.Tensor:
    shape = tf.shape(img)
    h, w = shape[0], shape[1]
    side = tf.minimum(h, w)
    oy = (h - side) // 2
    ox = (w - side) // 2
    return tf.image.crop_to_bounding_box(img, oy, ox, side, side)

def _box_blur_tf(x: tf.Tensor, k: int) -> tf.Tensor:
    # x: (H,W,1) float32
    kernel = tf.ones((k, k, 1, 1), tf.float32) / tf.cast(k * k, tf.float32)
    x4 = tf.expand_dims(x, 0)  # (1,H,W,1)
    y4 = tf.nn.depthwise_conv2d(x4, kernel, strides=[1,1,1,1], padding="SAME")
    return tf.squeeze(y4, 0)   # (H,W,1)

def _dilate(mask: tf.Tensor, k: int) -> tf.Tensor:
    m4 = tf.expand_dims(mask, 0)
    y4 = tf.nn.max_pool2d(m4, ksize=k, strides=1, padding="SAME")
    return tf.squeeze(y4, 0)

def _erode(mask: tf.Tensor, k: int) -> tf.Tensor:
    return 1.0 - _dilate(1.0 - mask, k)

def background_removal_fundus_tf(
    g01: tf.Tensor,
    blur_k: int = 51,
    thresh: float = 0.05,
    morph_k: int = 11,
    close_iters: int = 2
) -> tf.Tensor:
    """
    Fundus background removal:
    - smooth green channel
    - absolute threshold (keeps circular fundus)
    - morphology close to fill disk
    """
    # smooth
    g_blur = _box_blur_tf(g01, blur_k)

    # absolute threshold (key difference)
    mask = tf.cast(g_blur > thresh, tf.float32)

    # fill fundus disk
    for _ in range(close_iters):
        mask = _erode(_dilate(mask, morph_k), morph_k)

    return tf.clip_by_value(mask, 0.0, 1.0)


def local_contrast_gpu(gray01: tf.Tensor, kernel_size: int = 31, clip_std: float = 3.0, eps: float = 1e-5) -> tf.Tensor:
    """
    CLAHE-like local contrast normalization (GPU):
    computes local mean/std via depthwise conv and normalizes, clips, rescales.
    """
    k = kernel_size
    kernel = tf.ones((k, k, 1, 1), tf.float32) / tf.cast(k * k, tf.float32)

    x = tf.expand_dims(gray01, 0)  # (1,H,W,1)
    mean = tf.nn.depthwise_conv2d(x, kernel, [1,1,1,1], "SAME")
    mean2 = tf.nn.depthwise_conv2d(x * x, kernel, [1,1,1,1], "SAME")
    var = tf.maximum(mean2 - mean * mean, 0.0)
    std = tf.sqrt(var + eps)

    z = (x - mean) / std
    z = tf.clip_by_value(z, -clip_std, clip_std)
    z = (z / clip_std + 1.0) * 0.5
    z = tf.clip_by_value(z, 0.0, 1.0)

    return tf.squeeze(z, 0)  # (H,W,1)

def preprocess_core_tf(img_u8: tf.Tensor,
                       out_size=(300, 300),
                       bg_blur_k=51, bg_thresh=0.05, bg_morph_k=11,
                       contrast_k=31, contrast_clip=3.0) -> tf.Tensor:
    """
    central crop -> green -> background removal -> local contrast -> resize -> (G,G,G) in [0,255]
    returns float32 (300,300,3) range [0,255]
    """
    img_u8 = central_square_crop(img_u8)       # uint8
    g = img_u8[:, :, 1:2]                      # uint8 (H,W,1)
    g01 = tf.cast(g, tf.float32) / 255.0       # [0,1]

    # background mask from raw green
    mask = background_removal_fundus_tf(
    g01, blur_k=bg_blur_k, thresh=bg_thresh, morph_k=bg_morph_k
    )

    # local contrast (CLAHE-like) then apply mask
    g01 = local_contrast_gpu(g01, kernel_size=contrast_k, clip_std=contrast_clip)
    g01 = g01 * mask

    g01 = tf.image.resize(g01, out_size, method="area")   # (300,300,1)
    x = tf.repeat(g01, repeats=3, axis=-1)                # (300,300,3)
    return x * 255.0


# 3) Data augmentation (train only) — ενιαίο
def data_augmentation_tf(x_0_255: tf.Tensor) -> tf.Tensor:
    """
    x_0_255: float32 (H,W,3) in [0,255]
    vertical flip + rotate(0/90/180/270) + brightness/contrast
    """
    x = tf.image.random_flip_up_down(x_0_255)

    k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    x = tf.image.rot90(x, k=k)

    x01 = tf.clip_by_value(x / 255.0, 0.0, 1.0)
    x01 = tf.image.random_brightness(x01, max_delta=0.10)
    x01 = tf.image.random_contrast(x01, lower=0.90, upper=1.10)
    x = tf.clip_by_value(x01, 0.0, 1.0) * 255.0

    return x



# 4) Final mapping for EfficientNet pretrained
def to_effnet_input(x_0_255: tf.Tensor) -> tf.Tensor:
    x = effnet_preprocess_input(tf.cast(x_0_255, tf.float32))
    return x



# 5) tf.data Dataset builders
def make_dataset(paths, labels, batch_size=16, training=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(2048, reshuffle_each_iteration=True)

    def _map_fn(p, y):
        img = decode_png(p)
        x = preprocess_core_tf(img)       # float [0,255]
        if training:
            x = data_augmentation_tf(x)   # aug only train
        x = to_effnet_input(x)            # final for EfficientNet
        return x, y

    ds = ds.map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(X_train, y_train, batch_size=16, training=True)
val_ds   = make_dataset(X_val,   y_val,   batch_size=16, training=False)
test_ds  = make_dataset(X_test,  y_test,  batch_size=16, training=False)

print("Datasets ready.")

## Output : Split sizes: 24588 5269 5269

In [2]:
### **Visual Debugger: Preprocessing Pipeline Steps & Stage Analysis**

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf


# ----------------------------
# Helpers: save tensors as PNG
# ----------------------------
def _to_u8_3ch(img) -> tf.Tensor:
    """
    Accepts:
      - uint8 [0..255] (H,W,3) or (H,W,1)
      - float32 [0..1] (H,W,1/3)
      - float32 [0..255] (H,W,1/3)
    Returns uint8 (H,W,3)
    """
    img = tf.convert_to_tensor(img)

    if img.dtype.is_floating:
        # detect range: assume if max<=1.5 -> [0,1] else [0,255]
        mx = tf.reduce_max(img)
        img = tf.cond(mx <= 1.5, lambda: img * 255.0, lambda: img)
        img = tf.cast(tf.clip_by_value(img, 0.0, 255.0), tf.uint8)
    else:
        img = tf.cast(img, tf.uint8)

    # ensure 3 channels
    if tf.shape(img)[-1] == 1:
        img = tf.repeat(img, repeats=3, axis=-1)
    return img

def save_png(img, out_path: str):
    img_u8 = _to_u8_3ch(img)
    png = tf.image.encode_png(img_u8)
    tf.io.write_file(out_path, png)

def decode_png(path: tf.Tensor) -> tf.Tensor:
    b = tf.io.read_file(path)
    return tf.image.decode_png(b, channels=3)  # uint8


# Preprocess building blocks
def central_square_crop(img: tf.Tensor) -> tf.Tensor:
    shape = tf.shape(img)
    h, w = shape[0], shape[1]
    side = tf.minimum(h, w)
    oy = (h - side) // 2
    ox = (w - side) // 2
    return tf.image.crop_to_bounding_box(img, oy, ox, side, side)

def _box_blur_tf(x: tf.Tensor, k: int) -> tf.Tensor:
    kernel = tf.ones((k, k, 1, 1), tf.float32) / tf.cast(k * k, tf.float32)
    x4 = tf.expand_dims(x, 0)  # (1,H,W,1)
    y4 = tf.nn.depthwise_conv2d(x4, kernel, strides=[1,1,1,1], padding="SAME")
    return tf.squeeze(y4, 0)   # (H,W,1)

def _dilate(mask: tf.Tensor, k: int) -> tf.Tensor:
    m4 = tf.expand_dims(mask, 0)
    y4 = tf.nn.max_pool2d(m4, ksize=k, strides=1, padding="SAME")
    return tf.squeeze(y4, 0)

def _erode(mask: tf.Tensor, k: int) -> tf.Tensor:
    return 1.0 - _dilate(1.0 - mask, k)

def background_removal_fundus_tf(
    g01: tf.Tensor,
    blur_k: int = 51,
    thresh: float = 0.05,
    morph_k: int = 11,
    close_iters: int = 2
) -> tf.Tensor:
    """
    Fundus mask using absolute threshold after blur:
      mask = blur(g01) > thresh
    """
    g_blur = _box_blur_tf(g01, blur_k)
    mask = tf.cast(g_blur > thresh, tf.float32)
    for _ in range(close_iters):
        mask = _erode(_dilate(mask, morph_k), morph_k)  # close
    return tf.clip_by_value(mask, 0.0, 1.0)

def local_contrast_gpu(gray01: tf.Tensor, kernel_size: int = 31, clip_std: float = 3.0, eps: float = 1e-5) -> tf.Tensor:
    """
    CLAHE-like local contrast normalization (GPU friendly).
    gray01: float32 [0,1], (H,W,1)
    """
    k = kernel_size
    kernel = tf.ones((k, k, 1, 1), tf.float32) / tf.cast(k * k, tf.float32)

    x = tf.expand_dims(gray01, 0)  # (1,H,W,1)
    mean = tf.nn.depthwise_conv2d(x, kernel, [1,1,1,1], "SAME")
    mean2 = tf.nn.depthwise_conv2d(x * x, kernel, [1,1,1,1], "SAME")
    var = tf.maximum(mean2 - mean * mean, 0.0)
    std = tf.sqrt(var + eps)

    z = (x - mean) / std
    z = tf.clip_by_value(z, -clip_std, clip_std)
    z = (z / clip_std + 1.0) * 0.5
    z = tf.clip_by_value(z, 0.0, 1.0)

    return tf.squeeze(z, 0)  # (H,W,1)


# Main: visualize stages
def show_preprocess_stages(
    image_path: str,
    out_dir: str = "./debug_stages",
    out_size=(300, 300),
    # background removal params
    bg_blur_k=51, bg_thresh=0.05, bg_morph_k=11, bg_close_iters=2,
    # contrast params
    contrast_k=31, contrast_clip=3.0
):
    """
    Saves stage images to out_dir and returns a pandas Styler that displays them in order.
    """
    os.makedirs(out_dir, exist_ok=True)

    p = tf.constant(image_path)
    orig = decode_png(p)                       # uint8 (H,W,3)
    crop = central_square_crop(orig)           # uint8

    green_u8 = crop[:, :, 1:2]                 # uint8 (H,W,1)
    green01 = tf.cast(green_u8, tf.float32) / 255.0

    mask = background_removal_fundus_tf(
        green01, blur_k=bg_blur_k, thresh=bg_thresh, morph_k=bg_morph_k, close_iters=bg_close_iters
    )  # float (H,W,1) [0,1]

    bg_removed01 = green01 * mask              # float [0,1]
    contrast01 = local_contrast_gpu(green01, kernel_size=contrast_k, clip_std=contrast_clip)
    contrast_bg01 = contrast01 * mask

    resized01 = tf.image.resize(contrast_bg01, out_size, method="area")  # (300,300,1)
    resized_u8 = tf.cast(tf.clip_by_value(resized01 * 255.0, 0.0, 255.0), tf.uint8)  # for viewing

    # Save stages
    base = os.path.splitext(os.path.basename(image_path))[0]
    paths = []

    def _save(stage_name, tensor):
        fname = f"{base}_{stage_name}.png"
        fpath = os.path.join(out_dir, fname)
        save_png(tensor, fpath)
        paths.append((stage_name, fpath))

    _save("01_original", orig)
    _save("02_central_crop", crop)
    _save("03_green_channel", green_u8)
    _save("04_mask", mask)                 # will be shown as grayscale 3ch
    _save("05_bg_removed", bg_removed01)
    _save("06_contrast", contrast01)
    _save("07_contrast_bg", contrast_bg01)
    _save("08_resized_300", resized_u8)

In [3]:
### **Utility Function: Robust Tensor-to-PNG Exporter**

In [ ]:
import tensorflow as tf

def save_u8_png(img_0_255: tf.Tensor, path: str):
    
    img = tf.convert_to_tensor(img_0_255)

    # αν είναι float [0,1] → [0,255]
    if img.dtype.is_floating:
        mx = tf.reduce_max(img)
        img = tf.cond(
            mx <= 1.5,
            lambda: img * 255.0,
            lambda: img
        )
        img = tf.cast(tf.clip_by_value(img, 0.0, 255.0), tf.uint8)
    else:
        img = tf.cast(img, tf.uint8)

    # if 1 channel → 3 channel
    if img.shape.rank == 3 and img.shape[-1] == 1:
        img = tf.repeat(img, repeats=3, axis=-1)

    png = tf.image.encode_png(img)
    tf.io.write_file(path, png)


In [ ]:
### **Binary Classification Setup: Label Binarization & Dataset Pipelines**

In [ ]:
# Binary labels
y_train_bin = (y_train > 0).astype(np.int32)
y_val_bin   = (y_val   > 0).astype(np.int32)
y_test_bin  = (y_test  > 0).astype(np.int32)

# tf.data datasets
BATCH = 32 #32,62,128 or more

train_ds_bin = make_dataset(X_train, y_train_bin, batch_size=BATCH, training=True)
val_ds_bin   = make_dataset(X_val,   y_val_bin,   batch_size=BATCH, training=False)
test_ds_bin  = make_dataset(X_test,  y_test_bin,  batch_size=BATCH, training=False)


In [ ]:
### **Severity Grading Pipeline: Filtering & Re-labeling for Diseased-Only Subsets**

In [ ]:
# μάσκες για diseased
mask_train = y_train > 0
mask_val   = y_val   > 0
mask_test  = y_test  > 0

# paths + labels (1–4 -> 0–3)
X_train_dis = X_train[mask_train]
y_train_dis = (y_train[mask_train] - 1).astype(np.int32)

X_val_dis = X_val[mask_val]
y_val_dis = (y_val[mask_val] - 1).astype(np.int32)

X_test_dis = X_test[mask_test]
y_test_dis = (y_test[mask_test] - 1).astype(np.int32)

# tf.data datasets
train_ds_dis = make_dataset(X_train_dis, y_train_dis, batch_size=BATCH, training=True)
val_ds_dis   = make_dataset(X_val_dis,   y_val_dis,   batch_size=BATCH, training=False)
test_ds_dis  = make_dataset(X_test_dis,  y_test_dis,  batch_size=BATCH, training=False)

In [ ]:
### **Performance Optimization: Cached Data Pipeline with Decoupled Preprocessing & Augmentation**

In [ ]:
def make_dataset_fast(paths, labels, batch_size=16, training=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(2048, reshuffle_each_iteration=True)

    # 1) preprocess only (deterministic)
    def _pre(p, y):
        img = decode_png(p)
        x = preprocess_core_tf(img)          # float [0,255] BEFORE preprocess_input
        return x, y

    ds = ds.map(_pre, num_parallel_calls=tf.data.AUTOTUNE)

    # cache here
    ds = ds.cache()

    # 2) augmentation only for train
    if training:
        ds = ds.map(lambda x, y: (data_augmentation_tf(x), y),
                    num_parallel_calls=tf.data.AUTOTUNE)

    # 3) EfficientNet preprocess_input
    ds = ds.map(lambda x, y: (to_effnet_input(x), y),
                num_parallel_calls=tf.data.AUTOTUNE)

    # options for throughput
    options = tf.data.Options()
    options.experimental_deterministic = False
    ds = ds.with_options(options)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
import tensorflow as tf

print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Logical GPUs:", tf.config.list_logical_devices("GPU"))


In [ ]:
## **=== Τεχνικές Προδιαγραφές Συστήματος ===**

In [ ]:
import os
import platform
import psutil
import tensorflow as tf

print("=== Τεχνικές Προδιαγραφές Συστήματος ===")

# 1. GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        gpu_details = tf.config.experimental.get_device_details(gpus[0])
        print(f"GPU: {gpu_details.get('device_name', 'Unknown GPU')}")
    except:
        print("GPU: Εντοπίστηκε, αλλά δεν ήταν δυνατή η ανάγνωση του ονόματος.")
else:
    print("GPU: Δεν βρέθηκε κάρτα γραφικών.")

# 2. CPU check (Επεξεργαστής)
print(f"CPU: {platform.processor()}")
print(f"Φυσικοί Πυρήνες: {psutil.cpu_count(logical=False)}")
print(f"Λογικοί Πυρήνες (vCPUs): {psutil.cpu_count(logical=True)}")

# 3. RAM check
ram = psutil.virtual_memory().total / (1024**3)
print(f"Μνήμη RAM: {ram:.2f} GB")

# 4. Software Check
print(f"Λειτουργικό Σύστημα: {platform.system()} {platform.release()}")
print(f"Έκδοση TensorFlow: {tf.__version__}")
##Output :=== Τεχνικές Προδιαγραφές Συστήματος ===
#GPU: NVIDIA RTX 6000 Ada Generation
#CPU: x86_64
#Φυσικοί Πυρήνες: 18
#Λογικοί Πυρήνες (vCPUs): 18
#Μνήμη RAM: 62.27 GB
#Λειτουργικό Σύστημα: Linux 6.12.0-124.31.1.el10_1.x86_64
#Έκδοση TensorFlow: 2.20.0

In [ ]:
### **Deployment Optimization: Parallel Preprocessing, RAM Caching & GPU Prefetching**

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

# 1. New Preprocessing Function (For data that is ALREADY images)
def preprocess_function(image, label):
    # Ensure image is float32 for training
    image = tf.cast(image, tf.float32)
    # Normalize pixel values to [0, 1] range
    image = image / 255.0
    return image, label

# 2. Apply the Optimization Pipeline
# map:      Apply the normalization in parallel
train_ds = train_ds.map(preprocess_function, num_parallel_calls=AUTOTUNE)

# cache:    Keep images in RAM after the first epoch (HUGE speedup for small datasets)
train_ds = train_ds.cache()

# prefetch: Prepares the next batch while the GPU works (The magic speed fix)
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
### **Binary Model Training: Transfer Learning & Strategic Fine-Tuning with Mixed Precision**

In [ ]:
import numpy as np
import tensorflow as tf
from collections import Counter
from sklearn.metrics import confusion_matrix, accuracy_score

from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

mixed_precision.set_global_policy("mixed_float16")

def compute_class_weights(y):
    c = Counter(y)
    n = len(y)
    k = len(c)
    return {cls: n / (k * cnt) for cls, cnt in c.items()}

def build_effnetb3_binary(dropout=0.3):
    inp = layers.Input(shape=(300,300,3))
    base = EfficientNetB3(include_top=False, weights="imagenet", input_tensor=inp)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return models.Model(inp, out), base

def trainable_params_count(model):
    return int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))

# labels binary
y_train_bin = (y_train > 0).astype(np.int32)
y_val_bin   = (y_val   > 0).astype(np.int32)
y_test_bin  = (y_test  > 0).astype(np.int32)

cw_bin = compute_class_weights(y_train_bin)

model_bin, base_bin = build_effnetb3_binary(dropout=0.3)

callbacks_bin = [
    ModelCheckpoint("stageA_best.keras", monitor="val_recall", mode="max", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_recall", mode="max", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor="val_recall", mode="max", patience=4, restore_best_weights=True, verbose=1),
]

# Phase 1: frozen
model_bin.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

model_bin.fit(
    train_ds_bin,
    validation_data=val_ds_bin,
    epochs=8,
    class_weight=cw_bin,
    callbacks=callbacks_bin,
    verbose=1
)

# Phase 2: fine-tune
base_bin.trainable = True
for layer in base_bin.layers[:-60]:
    layer.trainable = False

model_bin.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

model_bin.fit(
    train_ds_bin,
    validation_data=val_ds_bin,
    epochs=12,
    class_weight=cw_bin,
    callbacks=callbacks_bin,
    verbose=1
)

# Evaluation + confusion matrix + trainable params
p_test = model_bin.predict(test_ds_bin, verbose=0).ravel()
y_pred = (p_test >= 0.5).astype(np.int32)

cm = confusion_matrix(y_test_bin, y_pred)
acc = accuracy_score(y_test_bin, y_pred)

print("\n===== STAGE A (Binary) Results =====")
print("Test Accuracy:", acc)
print("Confusion Matrix:\n", cm)
print("Trainable params:", trainable_params_count(model_bin))
model_bin.summary()


In [ ]:
### **Severity Grading:EfficientNetB3 Multiclass Training with Quadratic Weighted Kappa (QWK) Optimization**

In [ ]:
import numpy as np
import tensorflow as tf
from collections import Counter
from sklearn.metrics import confusion_matrix, quadratic_weighted_kappa, accuracy_score

from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

def compute_class_weights(y):
    c = Counter(y)
    n = len(y)
    k = len(c)
    return {cls: n / (k * cnt) for cls, cnt in c.items()}

class QWKCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, y_val_np, name="val_qwk", every=1):
        super().__init__()
        self.val_ds = val_ds
        self.y_val = y_val_np.astype(np.int32)
        self.name = name
        self.every = every

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if (epoch + 1) % self.every != 0:
            return

        preds = self.model.predict(self.val_ds, verbose=0)
        y_pred = np.argmax(preds, axis=1).astype(np.int32)
        qwk = quadratic_weighted_kappa(self.y_val, y_pred, weights="quadratic")
        logs[self.name] = qwk
        print(f" — {self.name}: {qwk:.5f}")

def build_effnetb3_multiclass(num_classes=4, dropout=0.35):
    inp = layers.Input(shape=(300,300,3))
    base = EfficientNetB3(include_top=False, weights="imagenet", input_tensor=inp)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    return models.Model(inp, out), base

def trainable_params_count(model):
    return int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))

# X_train_dis, y_train_dis, X_val_dis, y_val_dis, X_test_dis, y_test_dis
# train_ds_dis, val_ds_dis, test_ds_dis

cw_dis = compute_class_weights(y_train_dis)

model_dis, base_dis = build_effnetb3_multiclass(num_classes=4, dropout=0.35)

# QWK only here
qwk_cb = QWKCallback(val_ds_dis, y_val_dis, name="val_qwk", every=1)

callbacks_dis = [
    ModelCheckpoint("stageB_best.keras", monitor="val_qwk", mode="max", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_qwk", mode="max", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor="val_qwk", mode="max", patience=5, restore_best_weights=True, verbose=1),
    qwk_cb,
]

# Phase 1: frozen
model_dis.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_dis.fit(
    train_ds_dis,
    validation_data=val_ds_dis,
    epochs=10,
    class_weight=cw_dis,
    callbacks=callbacks_dis,
    verbose=1
)

# Phase 2: fine-tune
base_dis.trainable = True
for layer in base_dis.layers[:-80]:
    layer.trainable = False

model_dis.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_dis.fit(
    train_ds_dis,
    validation_data=val_ds_dis,
    epochs=15,
    class_weight=cw_dis,
    callbacks=callbacks_dis,
    verbose=1
)

# Evaluation + confusion matrix + trainable params
p_test = model_dis.predict(test_ds_dis, verbose=0)
y_pred = np.argmax(p_test, axis=1).astype(np.int32)

cm = confusion_matrix(y_test_dis, y_pred)
acc = accuracy_score(y_test_dis, y_pred)
qwk = quadratic_weighted_kappa(y_test_dis, y_pred, weights="quadratic")

print("\n===== STAGE B (Diseased 4-class) Results =====")
print("Test Accuracy:", acc)
print("Test QWK:", qwk)
print("Confusion Matrix:\n", cm)
print("Trainable params:", trainable_params_count(model_dis))
model_dis.summary()


In [ ]:
### **Data Exploration: Class Distribution Visualization (5-class vs. Binary)**

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

def plot_class_distributions(y_all, title_prefix=""):
    cnt = Counter(y_all)
    classes = [0,1,2,3,4]
    counts = [cnt.get(c, 0) for c in classes]

    # 5-class
    plt.figure()
    plt.bar(classes, counts)
    plt.xticks(classes)
    plt.xlabel("DR grade (0–4)")
    plt.ylabel("Count")
    plt.title(f"{title_prefix}Class distribution (5-class)")
    savefig("fig_class_distribution_5class.png")

    # binary
    y_bin = (y_all > 0).astype(int)
    cntb = Counter(y_bin)
    plt.figure()
    plt.bar(["Healthy (0)", "Diseased (1)"], [cntb.get(0,0), cntb.get(1,0)])
    plt.ylabel("Count")
    plt.title(f"{title_prefix}Class distribution (binary)")
    savefig("fig_class_distribution_binary.png")


plot_class_distributions(y_train, title_prefix="Train: ")

In [ ]:
### **Full Training Pipeline: Hierarchical DenseNet121 with Mixed Precision & Stage-Specific Optimization**

In [ ]:
import os
import numpy as np
import tensorflow as tf
from collections import Counter
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score

from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint



# Mixed precision (non mandatory — if there is strong GPU)
mixed_precision.set_global_policy("mixed_float16")



# Helpers
def compute_class_weights(y):
    c = Counter(y)
    n = len(y)
    k = len(c)
    return {cls: n / (k * cnt) for cls, cnt in c.items()}

def trainable_params_count(model):
    return int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))



# FAST dataset 
def decode_pp_png(path: tf.Tensor) -> tf.Tensor:
    b = tf.io.read_file(path)
    img = tf.image.decode_png(b, channels=3)         # uint8 (300,300,3)
    return tf.cast(img, tf.float32)                  # float [0,255]

def make_dataset_fast(paths, labels, batch_size=32, training=True):
    paths = np.array(paths, dtype=str)
    labels = np.array(labels, dtype=np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(4096, reshuffle_each_iteration=True)

    # --- If there is PRECOMPUTED 300x300:
    ds = ds.map(lambda p, y: (decode_pp_png(p), y), num_parallel_calls=tf.data.AUTOTUNE)


    if training:
        ds = ds.map(lambda x, y: (data_augmentation_tf(x), y),
                    num_parallel_calls=tf.data.AUTOTUNE)

    # DenseNet preprocess_input: expects float RGB in [0,255]
    ds = ds.map(lambda x, y: (densenet_preprocess_input(x), y),
                num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds



# DenseNet121 builder
def build_densenet121(num_classes, dropout=0.3):
    inp = layers.Input(shape=(300, 300, 3))
    base = DenseNet121(include_top=False, weights="imagenet", input_tensor=inp)
    base.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(dropout)(x)

    if num_classes == 1:
        out = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    else:
        out = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = models.Model(inp, out)
    return model, base



# Stage A: Binary (Healthy vs Diseased)
def run_stageA_densenet(train_paths, val_paths, test_paths, y_train, y_val, y_test,
                        batch_size=32, fine_tune_last=60,
                        epochs_frozen=6, epochs_ft=10):

    y_train_bin = (y_train > 0).astype(np.int32)
    y_val_bin   = (y_val   > 0).astype(np.int32)
    y_test_bin  = (y_test  > 0).astype(np.int32)

    cw_bin = compute_class_weights(y_train_bin)

    train_ds = make_dataset_fast(train_paths, y_train_bin, batch_size=batch_size, training=True)
    val_ds   = make_dataset_fast(val_paths,   y_val_bin,   batch_size=batch_size, training=False)
    test_ds  = make_dataset_fast(test_paths,  y_test_bin,  batch_size=batch_size, training=False)

    model, base = build_densenet121(num_classes=1, dropout=0.3)

    callbacks = [
        ModelCheckpoint("stageA_densenet_best.keras", monitor="val_recall", mode="max",
                        save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_recall", mode="max", factor=0.5, patience=2,
                          min_lr=1e-6, verbose=1),
        EarlyStopping(monitor="val_recall", mode="max", patience=4,
                      restore_best_weights=True, verbose=1),
    ]

    # ---- Phase 1: frozen
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.AUC(name="auc")]
    )

    print(">>> Stage A (DenseNet121) - Frozen training")
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_frozen,
              class_weight=cw_bin, callbacks=callbacks, verbose=1)

    # ---- Phase 2: fine-tune
    base.trainable = True
    # κρατάμε παγωμένα όλα εκτός από τα τελευταία fine_tune_last layers
    for layer in base.layers[:-fine_tune_last]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.AUC(name="auc")]
    )

    print(">>> Stage A (DenseNet121) - Fine-tuning")
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_ft,
              class_weight=cw_bin, callbacks=callbacks, verbose=1)

    # ---- Test eval
    p = model.predict(test_ds, verbose=0).ravel()
    pred = (p >= 0.5).astype(np.int32)

    cm = confusion_matrix(y_test_bin, pred)
    acc = accuracy_score(y_test_bin, pred)

    print("\n===== STAGE A (DenseNet121) Results =====")
    print("Test Accuracy:", acc)
    print("Confusion Matrix:\n", cm)
    print("Trainable params:", trainable_params_count(model))
    return model, cm, (y_test_bin, p, pred)



# Stage B: 4-class (μόνο diseased) + QWK κύριο metric
class QWKCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, y_val_np, name="val_qwk"):
        super().__init__()
        self.val_ds = val_ds
        self.y_val = y_val_np.astype(np.int32)
        self.name = name

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        pred = np.argmax(self.model.predict(self.val_ds, verbose=0), axis=1).astype(np.int32)
        qwk = cohen_kappa_score(self.y_val, pred, weights="quadratic")
        logs[self.name] = qwk
        print(f" — {self.name}: {qwk:.5f}")


def run_stageB_densenet(train_paths, val_paths, test_paths, y_train, y_val, y_test,
                        batch_size=32, fine_tune_last=80,
                        epochs_frozen=8, epochs_ft=12):

    # keep only diseased (1..4) then remap 0..3
    tr_mask = (y_train > 0)
    va_mask = (y_val   > 0)
    te_mask = (y_test  > 0)

    X_tr = np.array(train_paths)[tr_mask]
    X_va = np.array(val_paths)[va_mask]
    X_te = np.array(test_paths)[te_mask]

    y_tr = (y_train[tr_mask] - 1).astype(np.int32)   # 0..3
    y_va = (y_val[va_mask]   - 1).astype(np.int32)
    y_te = (y_test[te_mask]  - 1).astype(np.int32)

    cw = compute_class_weights(y_tr)

    train_ds = make_dataset_fast(X_tr, y_tr, batch_size=batch_size, training=True)
    val_ds   = make_dataset_fast(X_va, y_va, batch_size=batch_size, training=False)
    test_ds  = make_dataset_fast(X_te, y_te, batch_size=batch_size, training=False)

    model, base = build_densenet121(num_classes=4, dropout=0.4)

    qwk_cb = QWKCallback(val_ds, y_va, name="val_qwk")
    callbacks = [
        ModelCheckpoint("stageB_densenet_best.keras", monitor="val_qwk", mode="max",
                        save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_qwk", mode="max", factor=0.5, patience=2,
                          min_lr=1e-6, verbose=1),
        EarlyStopping(monitor="val_qwk", mode="max", patience=4,
                      restore_best_weights=True, verbose=1),
        qwk_cb,
    ]

    # Phase 1: frozen
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print(">>> Stage B (DenseNet121) - Frozen training")
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_frozen,
              class_weight=cw, callbacks=callbacks, verbose=1)

    # Phase 2: fine-tune
    base.trainable = True
    for layer in base.layers[:-fine_tune_last]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print(">>> Stage B (DenseNet121) - Fine-tuning")
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_ft,
              class_weight=cw, callbacks=callbacks, verbose=1)

    # Test eval
    probs = model.predict(test_ds, verbose=0)
    pred = np.argmax(probs, axis=1).astype(np.int32)

    cm = confusion_matrix(y_te, pred)
    acc = accuracy_score(y_te, pred)
    qwk = cohen_kappa_score(y_te, pred, weights="quadratic")

    print("\n===== STAGE B (DenseNet121) Results =====")
    print("Test Accuracy:", acc)
    print("Test QWK:", qwk)
    print("Confusion Matrix:\n", cm)
    print("Trainable params:", trainable_params_count(model))
    return model, cm, (y_te, pred, probs)



PP_DIR = "/home/jovyan/preprocessed_300"
X_train_pp = np.array([os.path.join(PP_DIR, os.path.basename(p)) for p in X_train])
X_val_pp   = np.array([os.path.join(PP_DIR, os.path.basename(p)) for p in X_val])
X_test_pp  = np.array([os.path.join(PP_DIR, os.path.basename(p)) for p in X_test])

# Stage A
model_A, cmA, stageA_outputs = run_stageA_densenet(
     X_train_pp, X_val_pp, X_test_pp, y_train, y_val, y_test,
     batch_size=32
)

# Stage B
model_B, cmB, stageB_outputs = run_stageB_densenet(
     X_train_pp, X_val_pp, X_test_pp, y_train, y_val, y_test,
     batch_size=32
)

In [ ]:
### **Hierarchical Classification Pipeline: Two-Stage ResNet50 Training with QWK Optimization**

In [ ]:
import os
import numpy as np
import tensorflow as tf
from collections import Counter
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score

from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


# Mixed precision
mixed_precision.set_global_policy("mixed_float16")



# Helpers
def compute_class_weights(y):
    c = Counter(y)
    n = len(y)
    k = len(c)
    return {cls: n / (k * cnt) for cls, cnt in c.items()}

def trainable_params_count(model):
    return int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))



# FAST dataset
def decode_pp_png(path: tf.Tensor) -> tf.Tensor:
    b = tf.io.read_file(path)
    img = tf.image.decode_png(b, channels=3)        # uint8 (300,300,3)
    return tf.cast(img, tf.float32)                 # float [0,255]

def make_dataset_fast(paths, labels, batch_size=32, training=True):
    paths = np.array(paths, dtype=str)
    labels = np.array(labels, dtype=np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(4096, reshuffle_each_iteration=True)

    # PRECOMPUTED 300x300:
    ds = ds.map(lambda p, y: (decode_pp_png(p), y), num_parallel_calls=tf.data.AUTOTUNE)


    if training:
        ds = ds.map(lambda x, y: (data_augmentation_tf(x), y),
                    num_parallel_calls=tf.data.AUTOTUNE)

    # ResNet preprocess_input: expects float RGB in [0,255]
    ds = ds.map(lambda x, y: (resnet_preprocess_input(x), y),
                num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


# ResNet50 builder
def build_resnet50(num_classes, dropout=0.3):
    inp = layers.Input(shape=(300, 300, 3))
    base = ResNet50(include_top=False, weights="imagenet", input_tensor=inp)
    base.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(dropout)(x)

    if num_classes == 1:
        out = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    else:
        out = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = models.Model(inp, out)
    return model, base



# Stage A: Binary (Healthy vs Diseased)
def run_stageA_resnet(train_paths, val_paths, test_paths, y_train, y_val, y_test,
                      batch_size=32, fine_tune_last=40,
                      epochs_frozen=6, epochs_ft=10):

    y_train_bin = (y_train > 0).astype(np.int32)
    y_val_bin   = (y_val   > 0).astype(np.int32)
    y_test_bin  = (y_test  > 0).astype(np.int32)

    cw_bin = compute_class_weights(y_train_bin)

    train_ds = make_dataset_fast(train_paths, y_train_bin, batch_size=batch_size, training=True)
    val_ds   = make_dataset_fast(val_paths,   y_val_bin,   batch_size=batch_size, training=False)
    test_ds  = make_dataset_fast(test_paths,  y_test_bin,  batch_size=batch_size, training=False)

    model, base = build_resnet50(num_classes=1, dropout=0.3)

    callbacks = [
        ModelCheckpoint("stageA_resnet_best.keras", monitor="val_recall", mode="max",
                        save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_recall", mode="max", factor=0.5, patience=2,
                          min_lr=1e-6, verbose=1),
        EarlyStopping(monitor="val_recall", mode="max", patience=4,
                      restore_best_weights=True, verbose=1),
    ]

    # Phase 1: frozen
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.AUC(name="auc")]
    )

    print(">>> Stage A (ResNet50) - Frozen training")
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_frozen,
              class_weight=cw_bin, callbacks=callbacks, verbose=1)

    # Phase 2: fine-tune
    base.trainable = True
    for layer in base.layers[:-fine_tune_last]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.AUC(name="auc")]
    )

   

    print(">>> Stage A (ResNet50) - Fine-tuning")
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_ft,
              class_weight=cw_bin, callbacks=callbacks, verbose=1)

    # Test eval
    p = model.predict(test_ds, verbose=0).ravel()
    pred = (p >= 0.5).astype(np.int32)

    cm = confusion_matrix(y_test_bin, pred)
    acc = accuracy_score(y_test_bin, pred)

    print("\n===== STAGE A (ResNet50) Results =====")
    print("Test Accuracy:", acc)
    print("Confusion Matrix:\n", cm)
    print("Trainable params:", trainable_params_count(model))
    return model, cm, (y_test_bin, p, pred)



# Stage B: 4-class (μόνο diseased) + QWK κύριο metric
class QWKCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, y_val_np, name="val_qwk"):
        super().__init__()
        self.val_ds = val_ds
        self.y_val = y_val_np.astype(np.int32)
        self.name = name

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        pred = np.argmax(self.model.predict(self.val_ds, verbose=0), axis=1).astype(np.int32)
        qwk = cohen_kappa_score(self.y_val, pred, weights="quadratic")
        logs[self.name] = qwk
        print(f" — {self.name}: {qwk:.5f}")


def run_stageB_resnet(train_paths, val_paths, test_paths, y_train, y_val, y_test,
                      batch_size=32, fine_tune_last=60,
                      epochs_frozen=8, epochs_ft=12):

    # diseased (1..4) remap in 0..3
    tr_mask = (y_train > 0)
    va_mask = (y_val   > 0)
    te_mask = (y_test  > 0)

    X_tr = np.array(train_paths)[tr_mask]
    X_va = np.array(val_paths)[va_mask]
    X_te = np.array(test_paths)[te_mask]

    y_tr = (y_train[tr_mask] - 1).astype(np.int32)  # 0..3
    y_va = (y_val[va_mask]   - 1).astype(np.int32)
    y_te = (y_test[te_mask]  - 1).astype(np.int32)

    cw = compute_class_weights(y_tr)

    train_ds = make_dataset_fast(X_tr, y_tr, batch_size=batch_size, training=True)
    val_ds   = make_dataset_fast(X_va, y_va, batch_size=batch_size, training=False)
    test_ds  = make_dataset_fast(X_te, y_te, batch_size=batch_size, training=False)

    model, base = build_resnet50(num_classes=4, dropout=0.4)

    qwk_cb = QWKCallback(val_ds, y_va, name="val_qwk")
    callbacks = [
        ModelCheckpoint("stageB_resnet_best.keras", monitor="val_qwk", mode="max",
                        save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_qwk", mode="max", factor=0.5, patience=2,
                          min_lr=1e-6, verbose=1),
        EarlyStopping(monitor="val_qwk", mode="max", patience=4,
                      restore_best_weights=True, verbose=1),
        qwk_cb,
    ]

    # Phase 1: frozen
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print(">>> Stage B (ResNet50) - Structure & Parameters")
    model.summary() # <--- ΕΔΩ ΤΟ ΒΑΖΕΙΣ

    print(">>> Stage B (ResNet50) - Frozen training")
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_frozen,
              class_weight=cw, callbacks=callbacks, verbose=1)

    # Phase 2: fine-tune
    base.trainable = True
    for layer in base.layers[:-fine_tune_last]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print(">>> Stage B (ResNet50) - Fine-tuning")
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_ft,
              class_weight=cw, callbacks=callbacks, verbose=1)

    # Test eval
    probs = model.predict(test_ds, verbose=0)
    pred = np.argmax(probs, axis=1).astype(np.int32)

    cm = confusion_matrix(y_te, pred)
    acc = accuracy_score(y_te, pred)
    qwk = cohen_kappa_score(y_te, pred, weights="quadratic")

    print("\n===== STAGE B (ResNet50) Results =====")
    print("Test Accuracy:", acc)
    print("Test QWK:", qwk)
    print("Confusion Matrix:\n", cm)
    print("Trainable params:", trainable_params_count(model))
    return model, cm, (y_te, pred, probs)



PP_DIR = "/home/jovyan/preprocessed_300"
X_train_pp = np.array([os.path.join(PP_DIR, os.path.basename(p)) for p in X_train])
X_val_pp   = np.array([os.path.join(PP_DIR, os.path.basename(p)) for p in X_val])
X_test_pp  = np.array([os.path.join(PP_DIR, os.path.basename(p)) for p in X_test])

# Stage A
model_A, cmA, outA = run_stageA_resnet(
     X_train_pp, X_val_pp, X_test_pp, y_train, y_val, y_test,
     batch_size=32
)

# Stage B
model_B, cmB, outB = run_stageB_resnet(
     X_train_pp, X_val_pp, X_test_pp, y_train, y_val, y_test,
     batch_size=32
)

In [ ]:
### **Hierarchical Learning Pipeline:(EfficientNetB3)Two-Stage Training (Binary & Multiclass) with Quadratic Kappa Optimization**

In [ ]:
import numpy as np
import tensorflow as tf
from collections import Counter
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score

from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint



# Mixed precision (μόνο αν έχεις GPU)
mixed_precision.set_global_policy("mixed_float16")


# Helpers

def compute_class_weights(y):
    c = Counter(y)
    n = len(y)
    k = len(c)
    return {cls: n / (k * cnt) for cls, cnt in c.items()}

def trainable_params_count(model):
    return int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))


# QWK callback ( Stage B)

class QWKCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, y_val_np, name="val_qwk", every=1):
        super().__init__()
        self.val_ds = val_ds
        self.y_val = y_val_np.astype(np.int32)
        self.name = name
        self.every = every

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if (epoch + 1) % self.every != 0:
            return

        preds = self.model.predict(self.val_ds, verbose=0)
        y_pred = np.argmax(preds, axis=1).astype(np.int32)
        qwk = cohen_kappa_score(self.y_val, y_pred, weights="quadratic")
        logs[self.name] = qwk
        print(f" — {self.name}: {qwk:.5f}")



# EfficientNetB3 builder
def build_effnetb3(num_classes, dropout=0.3):
    inp = layers.Input(shape=(300, 300, 3))
    base = EfficientNetB3(
        include_top=False,
        weights="imagenet",
        input_tensor=inp
    )
    base.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(dropout)(x)

    if num_classes == 1:
        out = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    else:
        out = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = models.Model(inp, out)
    return model, base



# STAGE A: Binary (Healthy vs Diseased)
# epochs: 8 (frozen) + 12 (fine-tune)
def run_stageA_effnet(train_ds_bin, val_ds_bin, test_ds_bin,
                      y_train, y_val, y_test,
                      fine_tune_last=60):

    y_train_bin = (y_train > 0).astype(np.int32)
    y_val_bin   = (y_val   > 0).astype(np.int32)
    y_test_bin  = (y_test  > 0).astype(np.int32)

    cw_bin = compute_class_weights(y_train_bin)

    model, base = build_effnetb3(num_classes=1, dropout=0.30)

    callbacks = [
        ModelCheckpoint("stageA_effnet_best.keras", monitor="val_recall",
                        mode="max", save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_recall", mode="max",
                          factor=0.5, patience=2, min_lr=1e-6, verbose=1),
        EarlyStopping(monitor="val_recall", mode="max",
                      patience=4, restore_best_weights=True, verbose=1),
    ]

    # Phase 1: frozen
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )

    print("\n=== Stage A (EfficientNetB3) — Frozen ===")
    model.fit(
        train_ds_bin,
        validation_data=val_ds_bin,
        epochs=8,
        class_weight=cw_bin,
        callbacks=callbacks,
        verbose=1,
    )

    # Phase 2: fine-tune
    base.trainable = True
    for layer in base.layers[:-fine_tune_last]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )

    print("\n=== Stage A (EfficientNetB3) — Fine-tuning ===")
    model.fit(
        train_ds_bin,
        validation_data=val_ds_bin,
        epochs=12,
        class_weight=cw_bin,
        callbacks=callbacks,
        verbose=1,
    )

    # Evaluation
    p = model.predict(test_ds_bin, verbose=0).ravel()
    y_pred = (p >= 0.5).astype(np.int32)

    cm = confusion_matrix(y_test_bin, y_pred)
    acc = accuracy_score(y_test_bin, y_pred)

    print("\n===== STAGE A (EfficientNetB3) Results =====")
    print("Test Accuracy:", acc)
    print("Confusion Matrix:\n", cm)
    print("Trainable params:", trainable_params_count(model))

    return model, cm



# STAGE B: Diseased 4-class
# epochs: 10 (frozen) + 15 (fine-tune)
def run_stageB_effnet(train_ds_dis, val_ds_dis, test_ds_dis,
                      y_train_dis, y_val_dis, y_test_dis,
                      fine_tune_last=80):

    cw_dis = compute_class_weights(y_train_dis)

    model, base = build_effnetb3(num_classes=4, dropout=0.35)

    qwk_cb = QWKCallback(val_ds_dis, y_val_dis, name="val_qwk", every=2)

    callbacks = [
        ModelCheckpoint("stageB_effnet_best.keras", monitor="val_qwk",
                        mode="max", save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_qwk", mode="max",
                          factor=0.5, patience=2, min_lr=1e-6, verbose=1),
        EarlyStopping(monitor="val_qwk", mode="max",
                      patience=5, restore_best_weights=True, verbose=1),
        qwk_cb,
    ]

    # Phase 1: frozen
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    print("\n=== Stage B (EfficientNetB3) — Frozen ===")
    model.fit(
        train_ds_dis,
        validation_data=val_ds_dis,
        epochs=10,
        class_weight=cw_dis,
        callbacks=callbacks,
        verbose=1,
    )

    # Phase 2: fine-tune
    base.trainable = True
    for layer in base.layers[:-fine_tune_last]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    print("\n=== Stage B (EfficientNetB3) — Fine-tuning ===")
    model.fit(
        train_ds_dis,
        validation_data=val_ds_dis,
        epochs=15,
        class_weight=cw_dis,
        callbacks=callbacks,
        verbose=1,
    )

    # Evaluation
    probs = model.predict(test_ds_dis, verbose=0)
    y_pred = np.argmax(probs, axis=1).astype(np.int32)

    cm = confusion_matrix(y_test_dis, y_pred)
    acc = accuracy_score(y_test_dis, y_pred)
    qwk = cohen_kappa_score(y_test_dis, y_pred, weights="quadratic")

    print("\n===== STAGE B (EfficientNetB3) Results =====")
    print("Test Accuracy:", acc)
    print("Test QWK:", qwk)
    print("Confusion Matrix:\n", cm)
    print("Trainable params:", trainable_params_count(model))

    return model, cm

# Stage A
model_A, cmA = run_stageA_effnet(
    train_ds_bin, val_ds_bin, test_ds_bin,
    y_train, y_val, y_test,
    fine_tune_last=60
)
print("Stage A done. CM:\n", cmA)
model_A.summary()

# Stage B
model_B, cmB = run_stageB_effnet(
    train_ds_dis, val_ds_dis, test_ds_dis,
    y_train_dis, y_val_dis, y_test_dis,
    fine_tune_last=80
)
print("Stage B done. CM:\n", cmB)
model_B.summary()

In [ ]:
## ** Accuracy**

In [ ]:
import pandas as pd

data = {
    'Μοντέλο': ['ResNet-50', 'DenseNet-121', 'EfficientNet-B3'],
    'Συνολικές Παράμετροι': [23595908, 8062596, 24300881],
    'Εκπαιδεύσιμες (Stage B)': [18077444, 1604228, 6755596],
    'Accuracy (Stage A)': [0.8009, 0.7863, 0.7144],
    'Accuracy (Stage B)': [0.5619, 0.5268, 0.5648],
    'QWK Score (Kappa)': [0.620, 0.614, 0.596]
}

df = pd.DataFrame(data)
df['Accuracy (Stage A)'] = df['Accuracy (Stage A)'].map('{:.2%}'.format)
df['Accuracy (Stage B)'] = df['Accuracy (Stage B)'].map('{:.2%}'.format)

print(df.to_string(index=False))

##Output :  Μοντέλο  Συνολικές Παράμετροι  Εκπαιδεύσιμες (Stage B) Accuracy (Stage A) Accuracy (Stage B)  QWK Score (Kappa)
#      ResNet-50              23595908                 18077444             80.09%             56.19%              0.620
#   DenseNet-121               8062596                  1604228             78.63%             52.68%              0.614
#EfficientNet-B3              24300881                  6755596             71.44%             56.48%              0.596

In [ ]:
## **Plats**

In [ ]:
import matplotlib.pyplot as plt

models = ['ResNet-50', 'DenseNet-121', 'EfficientNet-B3']
qwk_scores = [0.620, 0.614, 0.596]

plt.figure(figsize=(10, 6))
bars = plt.bar(models, qwk_scores, color=['#2ecc71', '#3498db', '#e74c3c'])

plt.title('Σύγκριση Quadratic Weighted Kappa (QWK) - Stage B', fontsize=14)
plt.ylabel('QWK Score', fontsize=12)
plt.ylim(0, 0.7) 


for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01, yval, ha='center', va='bottom', fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig('qwk_comparison.png')
plt.show()

In [ ]:
params = [18.07, 1.60, 6.75]
qwk = [0.620, 0.614, 0.596]
labels = ['ResNet-50', 'DenseNet-121', 'EfficientNet-B3']

plt.figure(figsize=(10, 6))
plt.scatter(params, qwk, s=200, color='darkblue')

for i, label in enumerate(labels):
    plt.annotate(label, (params[i], qwk[i]), xytext=(10, 10), textcoords='offset points', fontsize=12)

plt.title('Αποδοτικότητα: Εκπαιδεύσιμες Παράμετροι vs QWK', fontsize=14)
plt.xlabel('Εκπαιδεύσιμες Παράμετροι (εκατομμύρια)', fontsize=12)
plt.ylabel('QWK Score', fontsize=12)
plt.grid(True, alpha=0.3)
plt.savefig('efficiency_chart.png')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


# 1. ΕΙΣΑΓΩΓΗ ΔΕΔΟΜΕΝΩΝ (Συμπλήρωσε τα δικά σου νούμερα)


# --- EFFICIENTNET-B3 (Stage B - Πραγματικά δεδομένα από τα logs σου) ---
eff_train_acc = [0.365, 0.418, 0.432, 0.437, 0.447, 0.443, 0.448, 0.450, 0.450, 0.432, 0.471, 0.505, 0.525, 0.548, 0.553, 0.562, 0.572, 0.591, 0.599, 0.613]
eff_val_acc = [0.542, 0.500, 0.472, 0.491, 0.518, 0.450, 0.472, 0.517, 0.431, 0.474, 0.491, 0.454, 0.520, 0.497, 0.527, 0.538, 0.539, 0.545, 0.506, 0.529]
eff_test_acc = 0.5648 # Τελικό αποτέλεσμα

eff_train_loss = [1.204, 1.108, 1.094, 1.070, 1.064, 1.059, 1.054, 1.041, 1.043, 1.114, 0.994, 0.917, 0.864, 0.809, 0.787, 0.759, 0.725, 0.695, 0.670, 0.631]
eff_val_loss = [1.029, 1.028, 1.094, 1.064, 1.013, 1.067, 1.071, 0.987, 1.117, 1.108, 1.033, 1.099, 1.000, 1.062, 1.019, 1.000, 0.991, 0.977, 1.049, 1.032]
eff_test_loss = 0.985 # Προσέγγιση

# --- RESNET-50 (Placeholders - Συμπλήρωσε από τα logs σου) ---
res_train_acc = [0.40, 0.45, 0.50, 0.55, 0.58, 0.60] 
res_val_acc = [0.42, 0.48, 0.52, 0.54, 0.55, 0.54]   
res_test_acc = 0.5619
res_train_loss = [1.2, 1.1, 0.9, 0.8, 0.7, 0.6]      
res_val_loss = [1.1, 1.0, 0.95, 0.98, 0.97, 0.99]    
res_test_loss = 0.995

# --- DENSENET-121 (Placeholders - Συμπλήρωσε από τα logs σου) ---
dense_train_acc = [0.38, 0.42, 0.46, 0.50, 0.52, 0.55] 
dense_val_acc = [0.40, 0.44, 0.48, 0.49, 0.50, 0.51]   
dense_test_acc = 0.5268
dense_train_loss = [1.3, 1.1, 0.9, 0.8, 0.7, 0.65]     
dense_val_loss = [1.2, 1.1, 1.05, 1.08, 1.06, 1.07]    
dense_test_loss = 1.08


# 2. ΣΥΝΑΡΤΗΣΗ ΠΑΡΑΓΩΓΗΣ ΓΡΑΦΗΜΑΤΟΣ (3 σε 1)
def plot_performance(train_data, val_data, test_val, model_name, metric_name, filename):
    epochs = np.arange(1, len(train_data) + 1)
    plt.figure(figsize=(10, 6))
    
    # Σχεδίαση Training και Validation
    plt.plot(epochs, train_data, 'b-o', label=f'Training {metric_name}', linewidth=2, markersize=5)
    plt.plot(epochs, val_data, 'g-s', label=f'Validation {metric_name}', linewidth=2, markersize=5)
    
    # Σχεδίαση Testing ως οριζόντια διακεκομμένη γραμμή (Απαίτηση KF1)
    plt.axhline(y=test_val, color='r', linestyle='--', label=f'Final Testing {metric_name} ({test_val:.4f})', linewidth=2)
    
    plt.title(f'{model_name}: {metric_name} Evolution', fontsize=14, fontweight='bold')
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel(metric_name, fontsize=12)
    plt.legend(loc='best')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()


# 3. ΕΚΤΕΛΕΣΗ ΚΑΙ ΑΠΟΘΗΚΕΥΣΗ
# --- EfficientNet Plots ---
plot_performance(eff_train_acc, eff_val_acc, eff_test_acc, 'EfficientNet-B3', 'Accuracy', 'eff_acc.png')
plot_performance(eff_train_loss, eff_val_loss, eff_test_loss, 'EfficientNet-B3', 'Loss', 'eff_loss.png')

# --- ResNet Plots ---
plot_performance(res_train_acc, res_val_acc, res_test_acc, 'ResNet-50', 'Accuracy', 'res_acc.png')
plot_performance(res_train_loss, res_val_loss, res_test_loss, 'ResNet-50', 'Loss', 'res_loss.png')

# --- DenseNet Plots ---
plot_performance(dense_train_acc, dense_val_acc, dense_test_acc, 'DenseNet-121', 'Accuracy', 'dense_acc.png')
plot_performance(dense_train_loss, dense_val_loss, dense_test_loss, 'DenseNet-121', 'Loss', 'dense_loss.png')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Τα πραγματικά δεδομένα που έστειλες για το DenseNet-121
dense_cm_raw = np.array([
    [307, 52, 7, 0],
    [352, 285, 140, 17],
    [9, 24, 92, 6],
    [3, 16, 35, 52]
])

classes = ['Mild', 'Moderate', 'Severe', 'Proliferative']

def plot_densenet_cm(data, model_name, filename):
    # Κανονικοποίηση: Υπολογισμός ποσοστών (%) ανά γραμμή
    cm_perc = data.astype('float') / data.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    # Σχεδίαση Heatmap με ποσοστά αλλά και τα αρχικά νούμερα μέσα
    sns.heatmap(cm_perc, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=classes, yticklabels=classes,
                annot_kws={"size": 11, "weight": "bold"})
    
    plt.title(f'Confusion Matrix (Normalized): {model_name}', fontsize=14, pad=20)
    plt.ylabel('Actual Stage (Ground Truth)')
    plt.xlabel('Predicted Stage')
    
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

plot_densenet_cm(dense_cm_raw, 'DenseNet-121', 'cm_densenet_final.png')